# Apêndice 03b - `StandardScaler` em vez de `RobustScaler`

**Função de um escalonador.** Colocar colunas de unidades distintas em escala
comum, subtraindo uma medida de centro e dividindo por uma medida de
dispersão. Os dois candidatos adotam medidas diferentes:

- o `StandardScaler` usa **média e desvio padrão**, e deixa cada coluna com
  média 0 e desvio 1;
- o `RobustScaler` usa **mediana e IQR** (intervalo interquartil, a distância
  entre o primeiro e o terceiro quartil, ou seja, a largura da faixa que
  contém os 50% centrais). É dito robusto porque mediana e quartis não se
  alteram com valores extremos.

**Relevância da escolha.** O peso `1/√n` do notebook 03 só distribui um terço
da distância a cada bloco se cada coluna chegar com variância 1, o que o
`StandardScaler` garante por construção. O `RobustScaler` padroniza o IQR, de
modo que a variância resultante varia de coluna para coluna.

Foram comparadas três opções, todas com o mesmo `log1p` e o mesmo peso por
bloco:

| | Variáveis contínuas | Notas do IEGM |
|---|---|---|
| **A** | `RobustScaler` | `RobustScaler` (plano original) |
| **B** | `RobustScaler` | escala 1-5 original, apenas centrada |
| **C** | `StandardScaler` | `StandardScaler` (adotado) |

A comparação observa três aspectos em cada opção: o peso final de cada bloco,
o número de componentes necessárias para 80% da variância e a variável
dominante na segunda componente.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import RobustScaler, StandardScaler

# Mesma preparação do notebook 03: apontar para src/, ler a base e o
# dicionário e montar a lista de features ordenada por bloco.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
from config import BASE_FINAL_CSV, DICIONARIO_CSV

base = pd.read_csv(BASE_FINAL_CSV, dtype={"codigo_ibge": str})
dic = pd.read_csv(DICIONARIO_CSV)
bloco_de = dic.set_index("coluna")["bloco"]
ORDEM_BLOCOS = ["criminalidade", "socioeconomico", "gestao"]
features = sorted(dic.loc[dic["papel"] == "feature", "coluna"],
                  key=lambda c: (ORDEM_BLOCOS.index(bloco_de[c]), c))
modelagem = base[~base["flag_sem_iegm"]].reset_index(drop=True)

In [ ]:
# O log1p e o peso são iguais nas três opções: a única coisa que muda entre
# A, B e C é o escalonador. Por isso ficam prontos aqui fora.
log_cols = ([c for c in features if bloco_de[c] == "criminalidade"]
            + ["pib_percapita", "renda_domiciliar_mediana"])
ordinais = [c for c in features if bloco_de[c] == "gestao"]
continuas = [c for c in features if c not in ordinais]

X = modelagem[features].copy()
X[log_cols] = np.log1p(X[log_cols])

n_bloco = pd.Series([bloco_de[c] for c in features]).value_counts()
peso = pd.Series({c: 1 / np.sqrt(n_bloco[bloco_de[c]]) for c in features})

In [ ]:
def escalar(opcao):
    Z = pd.DataFrame(index=X.index, columns=features, dtype=float)
    if opcao == "A":
        # Tudo pelo RobustScaler, como estava no planejamento.
        Z[features] = RobustScaler().fit_transform(X[features])
    elif opcao == "B":
        # Contínuas no RobustScaler, mas as notas do IEGM mantendo a escala
        # original: só espremidas para 0..1 e centradas, sem dividir por
        # nenhuma medida de espalhamento.
        Z[continuas] = RobustScaler().fit_transform(X[continuas])
        Z[ordinais] = (X[ordinais] - 1) / 4        # 1..5 vira 0..1
        Z[ordinais] = Z[ordinais] - Z[ordinais].mean()
    else:
        # Opção C: tudo pelo StandardScaler.
        Z[features] = StandardScaler().fit_transform(X[features])
    return Z


# Mesma função do notebook 03: quanto da variância total cada bloco tem.
def orcamento(M):
    var = M.var(ddof=0)
    return (var.groupby(M.columns.map(bloco_de)).sum()
               .reindex(ORDEM_BLOCOS) / var.sum() * 100).round(1)

In [ ]:
linhas = {}
for opcao in ["A", "B", "C"]:
    W = escalar(opcao) * peso
    pca = PCA().fit(W)
    ev = pca.explained_variance_ratio_
    # Cargas da segunda componente, em módulo: serve para ver se alguma
    # variável sozinha está tomando conta dela.
    carga_pc2 = pd.Series(pca.components_[1], index=features).abs()
    linhas[opcao] = {
        # O ** espalha o dicionário de blocos como três colunas da tabela.
        **orcamento(W).to_dict(),
        "componentes p/ 80%": int(np.argmax(np.cumsum(ev) >= 0.80)) + 1,
        "maior carga na PC2": f"{carga_pc2.idxmax()} ({carga_pc2.max():.2f})",
    }
pd.DataFrame(linhas).T

## Leitura da tabela

**A - `RobustScaler` em todas as colunas.** Os blocos não se igualam
(34,3 / 35,6 / 30,1) e a segunda componente principal reduz-se praticamente a
uma única variável, `i_planejamento_ord`, com carga 0,93.

A causa aparece na última célula deste notebook. O IQR do `i_planejamento_ord`
é 0,33, porque 70% dos municípios receberam a mesma nota e os 50% centrais
quase não se dispersam, enquanto as demais dimensões do IEGM têm IQR entre 1,0
e 1,7. Como o `RobustScaler` divide pelo IQR, dividir por 0,33 equivale a
triplicar a coluna. O escalonador infla justamente a variável de menor poder
discriminante.

**B - manutenção da escala 1-5 das notas.** Sem divisão por medida de
dispersão, o bloco de gestão cai para 2,3% da distância, ou seja, desaparece
do cálculo. Como a gestão pública é o diferencial deste trabalho, a opção está
descartada de saída.

**C - `StandardScaler` em todas as colunas.** Cada bloco fica com 33,3%,
conforme previsto pelo peso `1/√n`, e nenhuma variável domina isoladamente uma
componente: a maior carga da PC2 é 0,42, contra 0,93 da opção A. São também as
12 componentes para 80% reportadas no notebook 03.

## Decisão

Adota-se o `StandardScaler`. A escolha original pelo `RobustScaler` visava
conter os valores extremos, função já cumprida pelo `log1p`: o notebook 02
mostrou que, após a transformação e desconsiderados os zeros, a assimetria das
taxas não ultrapassa 0,7 em módulo. Aplicar o `RobustScaler` sobre esse
resultado não acrescenta proteção e ainda rompe a igualdade entre os blocos,
decisão de projeto que o trabalho precisa sustentar.

In [ ]:
# A evidência por trás da leitura da opção A: o IQR é o divisor que o
# RobustScaler usa, então quanto menor ele for, mais a coluna é esticada.
iqr = (X[ordinais].quantile(0.75) - X[ordinais].quantile(0.25)).round(3)
iqr.rename("IQR (divisor do RobustScaler)").to_frame()